In [1]:
# Set project paths.
from pathlib import Path
import os
import sys

def find_project_root():
    current = Path.cwd()

    for folder in [current] + list(current.parents):
        if (folder / "Data").exists() and (folder / "Notebooks").exists():
            return folder

    raise FileNotFoundError("Could not find project root. Make sure Data and Notebooks folders exist.")

project_folder = find_project_root()
notebook_folder = project_folder / "Notebooks"

os.chdir(project_folder)

print("Project folder:", project_folder)
print("Notebook folder:", notebook_folder)

Project folder: /Users/mac/Dissertation/SEND-rebuild
Notebook folder: /Users/mac/Dissertation/SEND-rebuild/Notebooks


In [2]:
# Import packages.
import pandas as pd
import numpy as np

In [3]:
# Define generation repair helper.
class GenerationReconstructor:
    """Repairs a generation column on days where it reads far below
    what a matching "expected" series implies, while leaving every
    other column untouched."""

    @staticmethod
    def get_problem_dates(expected, actual, target_col, threshold):
        daily_actual = actual[[target_col]].resample("D").sum()
        daily_expected = expected.resample("D").sum()

        daily_ratio = daily_actual[target_col] / daily_expected["expected"]

        return daily_ratio[daily_ratio < threshold].index.date

    @staticmethod
    def calculate_performance_ratios(expected_clean, actual_clean, target_col):
        merged = expected_clean.merge(
            actual_clean[[target_col]],
            left_index=True,
            right_index=True,
        )

        merged["ratio"] = (
            merged[target_col] / merged["expected"]
        ).replace([np.inf, -np.inf], np.nan).fillna(0)

        monthly_ratios = merged.groupby(
            [merged.index.month, merged.index.time]
        )["ratio"].mean()

        overall_ratios = merged.groupby(merged.index.time)["ratio"].mean()

        return monthly_ratios, overall_ratios

    @classmethod
    def reconstruct_data(cls, actual, expected, target_col, threshold=0.01, default_ratio=0.85):
        prob_dates = cls.get_problem_dates(expected, actual, target_col, threshold)

        is_prob = pd.Series(actual.index.date, index=actual.index).isin(prob_dates)

        expected_clean = expected[~is_prob]
        actual_clean = actual[~is_prob]
        expected_problem = expected[is_prob]

        monthly_ratios, overall_ratios = cls.calculate_performance_ratios(
            expected_clean,
            actual_clean,
            target_col,
        )

        def calculate_repair(row):
            month = row.name.month
            time = row.name.time()

            ratio = monthly_ratios.get(
                (month, time),
                overall_ratios.get(time, default_ratio),
            )

            return row["expected"] * ratio

        repaired = actual.copy()

        if len(expected_problem) > 0:
            repaired.loc[expected_problem.index, target_col] = expected_problem.apply(
                calculate_repair, axis=1,
            )

        return repaired, prob_dates

In [4]:
# Define empirical wind-speed-to-generation curve.
class WindCurve:
    """Fits generation as a function of wind speed from clean data, so
    wind generation can be repaired the same way solar is: by comparing
    actual output against what the curve says is expected."""

    @staticmethod
    def fit(wind_speed, generation, bin_width=1.0):
        speed_bins = (wind_speed / bin_width).round() * bin_width

        return generation.groupby(speed_bins).median().sort_index()

    @staticmethod
    def predict(curve, wind_speed):
        return pd.Series(
            np.interp(wind_speed, curve.index.values, curve.values),
            index=wind_speed.index,
        )

    @classmethod
    def build_expected(cls, wind_speed, generation, bin_width=1.0):
        curve = cls.fit(wind_speed, generation, bin_width=bin_width)

        return pd.DataFrame(
            {"expected": cls.predict(curve, wind_speed)},
            index=wind_speed.index,
        )

In [5]:
# Load 2023 raw data.
deop_2023 = pd.read_csv(
    "Data/DEOP/2023_DEOP_Interp.csv",
    parse_dates=["DateTime"],
    index_col="DateTime",
)

solcast_2023 = pd.read_csv(
    "Data/Solcast/Solcast_2023.csv",
    parse_dates=["DateTime"],
    index_col="DateTime",
)

expected_2023 = pd.read_csv(
    "Data/DEOP/2023_Expected.csv",
    parse_dates=["DateTime"],
    index_col="DateTime",
)

print("2023 DEOP range:", deop_2023.index.min(), "to", deop_2023.index.max())
print("2023 Solcast range:", solcast_2023.index.min(), "to", solcast_2023.index.max())

2023 DEOP range: 2023-01-01 00:05:00 to 2023-12-31 23:55:00
2023 Solcast range: 2023-01-01 00:05:00 to 2023-12-31 23:55:00


In [6]:
# Repair 2023 solar data.
deop_2023_solar_fixed, solar_problem_dates_2023 = GenerationReconstructor.reconstruct_data(
    actual=deop_2023,
    expected=expected_2023,
    target_col="power-gen-pv-ave",
    threshold=0.01,
)

print("2023 solar problem days:", len(solar_problem_dates_2023))
print(solar_problem_dates_2023[:10])

2023 solar problem days: 22
[datetime.date(2023, 1, 28) datetime.date(2023, 1, 29)
 datetime.date(2023, 3, 27) datetime.date(2023, 4, 22)
 datetime.date(2023, 6, 20) datetime.date(2023, 7, 2)
 datetime.date(2023, 7, 3) datetime.date(2023, 7, 4)
 datetime.date(2023, 7, 5) datetime.date(2023, 7, 6)]


In [7]:
# Repair 2023 wind data.
expected_wind_2023 = WindCurve.build_expected(
    solcast_2023["wind_speed_100m"],
    deop_2023_solar_fixed["power-gen-wt-ave"],
)

deop_2023_repaired, wind_problem_dates_2023 = GenerationReconstructor.reconstruct_data(
    actual=deop_2023_solar_fixed,
    expected=expected_wind_2023,
    target_col="power-gen-wt-ave",
    threshold=0.1,
)

print("2023 wind problem days:", len(wind_problem_dates_2023))
print(wind_problem_dates_2023[:10])

2023 wind problem days: 11
[datetime.date(2023, 3, 27) datetime.date(2023, 4, 22)
 datetime.date(2023, 4, 23) datetime.date(2023, 4, 24)
 datetime.date(2023, 4, 25) datetime.date(2023, 6, 20)
 datetime.date(2023, 10, 3) datetime.date(2023, 10, 4)
 datetime.date(2023, 10, 29) datetime.date(2023, 10, 30)]


In [8]:
# Save 2023 repair.
deop_2023_repaired.to_csv("Data/DEOP/2023_DEOP_Repaired.csv")

print("Saved Data/DEOP/2023_DEOP_Repaired.csv")
print("NaNs remaining:", deop_2023_repaired.isna().sum().to_dict())

Saved Data/DEOP/2023_DEOP_Repaired.csv
NaNs remaining: {'power-con-ave': 0, 'power-gen-wt-ave': 0, 'power-gen-pv-ave': 0}


In [9]:
# Load 2022 raw data.
deop_2022 = pd.read_csv(
    "Data/DEOP/2022_DEOP_Interp.csv",
    parse_dates=["DateTime"],
    index_col="DateTime",
)

solcast_2022_full = pd.read_csv(
    "Data/Solcast/Solcast_2022.csv",
    parse_dates=["DateTime"],
    index_col="DateTime",
)

expected_2022_full = pd.read_csv(
    "Data/DEOP/2022_Expected.csv",
    parse_dates=["DateTime"],
    index_col="DateTime",
)

print("2022 DEOP range:", deop_2022.index.min(), "to", deop_2022.index.max())
print("2022 Solcast range (raw):", solcast_2022_full.index.min(), "to", solcast_2022_full.index.max())

2022 DEOP range: 2022-03-01 00:00:00 to 2022-12-31 23:55:00
2022 Solcast range (raw): 2022-01-01 00:05:00 to 2022-12-31 23:55:00


In [10]:
# Align 2022 weather data down to DEOP's real collection window.
#
# DEOP campus monitoring only started in March 2022, while the Solcast weather
# feed covers the full year. Padding DEOP up to match Solcast (the original
# approach) leaves Jan-Feb as NaN/0, which then gets flagged as "missing solar"
# and repaired with fabricated values for a period that was never monitored.
# Cropping the weather data down to DEOP's real window avoids inventing data.
solcast_2022 = solcast_2022_full.loc[deop_2022.index.min():deop_2022.index.max()]
expected_2022 = expected_2022_full.loc[deop_2022.index.min():deop_2022.index.max()]

print("2022 Solcast range (cropped):", solcast_2022.index.min(), "to", solcast_2022.index.max())
print("Index alignment matches DEOP:", solcast_2022.index.equals(deop_2022.index))

2022 Solcast range (cropped): 2022-03-01 00:00:00 to 2022-12-31 23:55:00
Index alignment matches DEOP: True


In [11]:
# Repair 2022 solar data.
deop_2022_solar_fixed, solar_problem_dates_2022 = GenerationReconstructor.reconstruct_data(
    actual=deop_2022,
    expected=expected_2022,
    target_col="power-gen-pv-ave",
    threshold=0.01,
)

print("2022 solar problem days:", len(solar_problem_dates_2022))
print(solar_problem_dates_2022[:10])

2022 solar problem days: 29
[datetime.date(2022, 3, 1) datetime.date(2022, 3, 2)
 datetime.date(2022, 3, 3) datetime.date(2022, 3, 4)
 datetime.date(2022, 3, 5) datetime.date(2022, 3, 6)
 datetime.date(2022, 3, 7) datetime.date(2022, 3, 8)
 datetime.date(2022, 3, 9) datetime.date(2022, 3, 10)]


In [12]:
# Repair 2022 wind data.
expected_wind_2022 = WindCurve.build_expected(
    solcast_2022["wind_speed_100m"],
    deop_2022_solar_fixed["power-gen-wt-ave"],
)

deop_2022_repaired, wind_problem_dates_2022 = GenerationReconstructor.reconstruct_data(
    actual=deop_2022_solar_fixed,
    expected=expected_wind_2022,
    target_col="power-gen-wt-ave",
    threshold=0.1,
)

print("2022 wind problem days:", len(wind_problem_dates_2022))
print(wind_problem_dates_2022[:10])

2022 wind problem days: 22
[datetime.date(2022, 3, 1) datetime.date(2022, 3, 2)
 datetime.date(2022, 3, 3) datetime.date(2022, 3, 4)
 datetime.date(2022, 3, 5) datetime.date(2022, 3, 6)
 datetime.date(2022, 3, 7) datetime.date(2022, 3, 8)
 datetime.date(2022, 3, 9) datetime.date(2022, 3, 10)]


In [13]:
# Save 2022 repair.
deop_2022_repaired.to_csv("Data/DEOP/2022_DEOP_Repaired.csv")

print("Saved Data/DEOP/2022_DEOP_Repaired.csv")
print("NaNs remaining:", deop_2022_repaired.isna().sum().to_dict())

Saved Data/DEOP/2022_DEOP_Repaired.csv
NaNs remaining: {'power-con-ave': 0, 'power-gen-wt-ave': 0, 'power-gen-pv-ave': 0}
